In [ ]:
import os
import json

from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_cohere import CohereRerank
from langchain_huggingface import ChatHuggingFace
from langchain_huggingface.llms import HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers.fix import OutputFixingParser
from pydantic import BaseModel
from typing import List

from dotenv import load_dotenv

In [25]:
load_dotenv()

True

In [16]:
DATA_PATH = "../data"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 0
VECTOR_STORE_DIR = "db/chroma_db"

## Load the data

In [ ]:
with open(f"{DATA_PATH}/documents.jsonl", "r") as f:
    for idx, line in enumerate(f):
        obj = json.loads(line)
        with open(f"../data/{idx}.txt", "w") as txt_file:
            txt_file.write(obj['content'])

In [ ]:
loader = DirectoryLoader(
        DATA_PATH,
        glob="*.txt", # load only the text files
        loader_cls=TextLoader) # use the TextLoader to load the text files (since we are working with text files, there are more loader classes to choose from)

documents = loader.load()

if len(documents) == 0:
        raise FileNotFoundError(f"No documents found in the directory {DATA_PATH}.")

print(f"Loaded {len(documents)} documents from {DATA_PATH}!")

## Chunking

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = text_splitter.split_documents(documents)

if len(chunks) == 0:
    raise ValueError(f"No chunks were generated from the documents.\nChunk size: {CHUNK_SIZE}\nChunk overlap: {CHUNK_OVERLAP}")    
print(f"Success! Split {len(documents)} documents into {len(chunks)} chunks.")

In [ ]:
print("Embedding the chunks and storing them in a vector database (ChromaDB)")

# https://modal.com/blog/embedding-models-article
embedding_model = HuggingFaceEmbeddings(model="intfloat/e5-large-v2")
# embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=1536)

print("Creating a ChromaDB vector store....")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=VECTOR_STORE_DIR,
    collection_metadata={"hnsw:space": "cosine"}
)
print("Finished adding documents to the vector store.")

## Basic Retrieval

In [ ]:
query = "What do I need to do when I first move to the Netherlands?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
relevant_docs = retriever.invoke(query)

# # Debug
# print(f"User Query : {query}")
# print("--- Context ---")
# for i, doc in enumerate(relevant_docs):
#     print(f"Document {i}\n{doc.page_content}")
#     print("-"*100)

In [33]:
prompt = f"""Based on the following documents, please answer this question : {query}

Documents : 
{chr(10).join([f" - {doc.page_content}" for doc in relevant_docs])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer the question".
"""


In [ ]:
hf_endpoint = HuggingFaceEndpoint(
    model="meta-llama/Llama-3.1-8B-Instruct", # you have to make sure that this model has an InferenceProvider on the HuggingFace Website.
    task="conversational",
    max_new_tokens=200,
    temperature=0.7,
    provider="auto"
)

llm = ChatHuggingFace(llm=hf_endpoint)

# If you have an OpenAI API key, you can directly use ChatOpenAI like this
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = [
    SystemMessage(content="You are a helpful assistant that answers questions based on the provided documents. If you can't find the answer in the documents, say 'I don't have enough information to answer the question'."),
    HumanMessage(content=prompt)
]

response = llm.invoke(messages)

print(response.content)

## Multi Query Search & RRF

In [ ]:
def reciprocal_rank_fusion(extracted_chunks, k=60):
    scores = {}
    for query_chunks in extracted_chunks:
        for i, doc in enumerate(query_chunks):
            if not doc.id in scores:
                scores[doc.id] = [doc, 1 / (k+i+1)]
            else:
                prev_score = scores[doc.id][-1]
                prev_score += (1/(k+i+1))
                scores[doc.id][-1] = prev_score
    
    sorted_chunks = dict(sorted(scores.items(), key=lambda item: item[1][1], reverse=True))
    return sorted_chunks

In [ ]:
class QueryVariations(BaseModel):
    queries: List[str]

base_parser = JsonOutputParser(pydantic_object=QueryVariations)

# This is the safety net. 
# If the JsonOutputParser fails, this parser doesn't just throw an error. 
# Instead, it takes the bad output, the error message, and the original prompt, and sends them back to the LLM saying: "You messed up the JSON. Here is the error. Please fix it."
# It effectively "self-heals" the data.
robust_parser = OutputFixingParser.from_llm(parser=base_parser, llm=llm)

# input_variables: These are the variables you provide every time you run the code (like the og_query).
# partial_variables: These are "pre-filled" variables. Instead of manually typing "Please return JSON..." every time, you use get_format_instructions(). 
# This dynamically injects the exact technical requirements derived from your Pydantic model into the prompt.
prompt = PromptTemplate(
    template="Generate THREE different variations of this query. \n{format_instructions}\nOriginal: {og_query}. Return 3 alternative queries that rephrase or approach the same question from different angles",
    input_variables=["og_query"],
    partial_variables={"format_instructions": robust_parser.get_format_instructions()},
)

# LangChain uses LCEL (LangChain Expression Language). The pipe operator (|) works exactly like a literal assembly line:
    # Input: You pass {"og_query": "..."}.
    # prompt: Takes your query, merges it with the format instructions, and creates one big string.
    # llm: Receives that string, thinks, and spits out a raw text response (potentially messy).
    # robust_parser: Receives the raw text. It tries to parse it; if it's broken, it triggers the "Fixing" logic to ensure you get a clean Python object back.
chain = prompt | llm | robust_parser

response = chain.invoke({"og_query": "What do I need to do when I first move to the Netherlands?" })
query_variations = response['queries']

retriever = vectorstore.as_retriever(search_kwargs={"k":5})

extracted_documents = []

for query in query_variations:
    relevant_docs = retriever.invoke(query)
    extracted_documents.append(relevant_docs)
    print(f"Documents found for query : {query}")
    for i, doc in enumerate(relevant_docs):
        print(f"        Document {i} - {doc.page_content[:300]}...")

    print("-"*100)

rrf_outputs = reciprocal_rank_fusion(extracted_documents)

## Hybrid Search

In [ ]:
bm25_retriever = BM25Retriever.from_documents(documents) # Keep in mind that these are LangChain documents, not chunks
bm25_retriever.k = 3 

In [ ]:
query = "Zoekjaar"
test_docs = bm25_retriever.invoke(query)

# # Debug
# for doc in test_docs:
#     print(doc)

### Combining Vector + Keyword Search

In [ ]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[retriever, bm25_retriever],
    weights=[0.5, 0.5] # this will be used in the RRF calculation
)

In [ ]:
query = "Zoekjaar requirements"
retrieved_chunks = hybrid_retriever.invoke(query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"Chunk {i} - {doc.page_content}")

In [ ]:
combined_input = f"""Based on the following documents, please answer this question : {query}

Documents : 
{chr(10).join([f" - {doc.page_content}" for doc in retrieved_chunks])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer the question".
"""

## Reranking

In [ ]:
reranker = CohereRerank(model="rerank-english-v3.0", top_n=10)
reranked_docs = reranker.compress_documents(retrieved_chunks, query)

In [ ]:
for i, doc in enumerate(reranked_docs, 1):
    print(f"Chunk {i} - {doc.page_content}")

# Then you would inject these chunks in the prompt and call the model normally as we did above.